# Inference + Evaluation — Fine-tuned VLM

Runs the fine-tuned LoRA adapter on the Romanian VQA test set and computes
accuracy / F1 / BLEU / ROUGE / BERTScore.

**Pipeline:**
1. Mount Drive, unzip images, load test JSON
2. Cache only the test images
3. Load the fine-tuned model with Unsloth in 4-bit
4. Generate predictions and stream them to a JSONL
5. Evaluate with several metrics + Romanian-aware text normalization

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install rouge-score bert-score nltk sentence-transformers scikit-learn
import torch; torch._dynamo.config.recompile_limit = 64

In [ ]:
import os
import json
import zipfile
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datasets import Dataset

## 2. Config

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Retranslation"
TEST_JSON   = "test_dataset_retranslated_gemma_bigmodel.json"
IMAGES_ZIP  = "/content/drive/My Drive/licenta/images.zip"
IMAGE_DIR   = "/content/data/images"

# Path to your saved fine-tuned model (LoRA + tokenizer) OR a HF hub repo id.
# Examples:
#   "/content/drive/My Drive/licenta/new_llm/qwen-big-retranlsate-100k"
#   "<your-hf-username>/<adapter-name>"
MODEL_PATH = "/content/drive/My Drive/licenta/new_llm/gemma4-big-retranlsate-100k"

# Where predictions will be appended (JSONL, one record per line)
SAVE_PATH = "/content/drive/My Drive/licenta/new_llm/save_json/predictions.jsonl"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

## 3. Unzip images + load test set

In [ ]:
os.makedirs("/content/data", exist_ok=True)
with zipfile.ZipFile(IMAGES_ZIP, "r") as z:
    z.extractall("/content/data")
print("Images extracted to", IMAGE_DIR)

In [ ]:
def load_dataset(json_name):
    path = os.path.join(DATASET_DIR, json_name)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame.from_dict(data, orient="index")
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'questionId'}, inplace=True)
    return df

test_df = load_dataset(TEST_JSON)
test_dataset = Dataset.from_pandas(test_df)
print(f"Test examples: {len(test_dataset)}")

## 4. Cache only the test images

In [ ]:
unique_image_ids = set(ex["imageId"] for ex in test_dataset)
image_cache = {}
for image_id in tqdm(unique_image_ids, desc="Caching test images"):
    path = os.path.join(IMAGE_DIR, f"{image_id}.jpg")
    try:
        image_cache[image_id] = Image.open(path).convert("RGB")
    except Exception as e:
        print(f"Failed to load {image_id}: {e}")
print(f"Cached {len(image_cache)} images")

## 5. Load the fine-tuned model

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name   = MODEL_PATH,
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)
print("Model loaded.")

## 6. Inference helper (Romanian, optional caption context)

In [ ]:
def run_inference(model, tokenizer, image, question, caption=None,
                  max_new_tokens=128, temperature=0.1, min_p=0.1):
    base_instruction = "Răspunde la următoarea întrebare despre imagine."
    if caption and isinstance(caption, str) and len(caption.strip()) > 1:
        full_text = (f"Descriere context: {caption}\n\n{base_instruction}\n\n"
                     f"Întrebare: {question}\nRăspuns:")
    else:
        full_text = f"{base_instruction}\n\nÎntrebare: {question}\nRăspuns:"

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": full_text},
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        image, input_text,
        return_tensors="pt", add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = temperature,
            min_p          = min_p,
            do_sample      = temperature > 0,
            use_cache      = True,
        )

    full = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return full.split("Răspuns:")[-1].strip()

## 7. Run inference (with resume)

In [ ]:
processed_keys = set()
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                processed_keys.add(json.loads(line)["questionId"])
            except Exception:
                continue
print(f"Already processed: {len(processed_keys)}")

with open(SAVE_PATH, "a", encoding="utf-8") as out_f:
    for ex in tqdm(test_dataset, desc="Inference"):
        qid = ex["questionId"]
        if qid in processed_keys:
            continue

        image = image_cache.get(ex["imageId"])
        if image is None:
            continue

        caption = ex.get("caption_llava")
        if caption is not None and pd.isna(caption):
            caption = None

        try:
            pred = run_inference(model, tokenizer, image, ex["question"], caption=caption)
        except Exception as e:
            print(f"Error on {qid}: {e}")
            continue

        record = {
            "questionId": qid,
            "imageId":    ex["imageId"],
            "question":   ex["question"],
            "gt_answer":  ex.get("answer_clean") or ex.get("answer", ""),
            "prediction": pred,
        }
        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        out_f.flush()
        processed_keys.add(qid)

## 8. Evaluation

Romanian-aware text normalization + a suite of metrics: accuracy, weighted F1,
BLEU, ROUGE-1/2/L, BERTScore, METEOR, and embedding cosine similarity.

In [ ]:
import re
import unicodedata
import nltk
from sklearn.metrics import accuracy_score, f1_score
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_scorer
from sentence_transformers import SentenceTransformer, util

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def clean_text(s):
    if not s:
        return ""
    s = unicodedata.normalize("NFC", str(s).lower().strip())
    s = re.sub(r"\s+", " ", s).strip().strip(".")
    return s

def clean_prediction(pred):
    m = re.search(r"Răspuns:\s*assistant\s*\n*\s*(.*)", pred, re.IGNORECASE)
    if m:
        return clean_text(m.group(1))
    return clean_text(pred)

In [ ]:
def evaluate_vqa(jsonl_path):
    preds = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                preds.append(json.loads(line))
            except Exception:
                continue

    gt   = [clean_text(p["gt_answer"])     for p in preds]
    pred = [clean_prediction(p["prediction"]) for p in preds]

    accuracy = accuracy_score(gt, pred)
    f1       = f1_score(gt, pred, average="weighted")
    bleu     = corpus_bleu(
        [[r.split()] for r in gt],
        [h.split() for h in pred],
        smoothing_function=SmoothingFunction().method1,
    )

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = [scorer.score(r, p) for r, p in zip(gt, pred)]
    def avg_rouge(k):
        return sum(s[k].fmeasure for s in rouge_scores) / len(rouge_scores)

    _, _, bert_f1 = bert_scorer(pred, gt, lang="ro", verbose=False)

    results = {
        "n_examples":    len(preds),
        "accuracy":      round(accuracy, 4),
        "f1_weighted":   round(f1, 4),
        "bleu":          round(bleu, 4),
        "rouge1":        round(avg_rouge("rouge1"), 4),
        "rouge2":        round(avg_rouge("rouge2"), 4),
        "rougeL":        round(avg_rouge("rougeL"), 4),
        "bertscore_f1":  round(bert_f1.mean().item(), 4),
    }
    print("Evaluation results:")
    for k, v in results.items():
        print(f"  {k:<15} {v}")
    return results

evaluate_vqa(SAVE_PATH)

## 9. Visualize predictions (optional)

In [ ]:
import matplotlib.pyplot as plt
import textwrap
import random

def visualize_predictions(jsonl_path, n_samples=3):
    preds = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                preds.append(json.loads(line))
            except Exception:
                continue

    samples = random.sample(preds, min(n_samples, len(preds)))
    for ex in samples:
        image = image_cache.get(ex["imageId"])
        if image is None:
            continue

        fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(12, 6),
                                              gridspec_kw={"width_ratios": [1.8, 1]})
        ax_img.imshow(image)
        ax_img.axis("off")

        gt   = clean_text(ex["gt_answer"])
        pred = clean_prediction(ex["prediction"])
        match_color = "green" if gt == pred else "red"

        wrapped_q = "\n".join(textwrap.wrap(ex["question"], width=35))
        ax_txt.axis("off")
        ax_txt.text(0, 0.95, "Question:", fontsize=12, fontweight="bold", va="top")
        ax_txt.text(0, 0.85, wrapped_q,   fontsize=11, va="top")
        ax_txt.text(0, 0.55, f"Gold: {gt}", fontsize=11, va="top")
        ax_txt.text(0, 0.45, f"Pred: {pred}", fontsize=11, va="top", color=match_color)
        plt.show()

visualize_predictions(SAVE_PATH, n_samples=3)